In [1]:
import os
import h5py
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler

loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'

encoder = load_model(os.path.join(loadPath, "models", "latent_encoder.keras"))

with open(os.path.join(loadPath, "models", "latent_autoencoder_scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

# ---------------------------
# load raw splits
# ---------------------------
def load_split(filename):
    f = h5py.File(os.path.join(loadPath, filename), "r")
    data = f['data'][:]
    tasks = f['tasks'][:]
    subjects = f['subjects'][:]
    f.close()
    return data, tasks, subjects

tr_data, ytr, tr_subjects = load_split("train1800_raw_EEG.h5")
val_data, yval, val_subjects = load_split("valid360_raw_EEG.h5")
ts_data, yts, ts_subjects = load_split("test360_raw_EEG.h5")

def preprocess(data):
    flat = np.squeeze(data).reshape((-1, 64))
    scaled = scaler.transform(flat)
    return scaled.reshape((-1, 640, 64, 1)).astype(np.float32)

x_train = preprocess(tr_data)
x_valid = preprocess(val_data)
x_test = preprocess(ts_data)

z_train = encoder.predict(x_train, batch_size=64)
z_valid = encoder.predict(x_valid, batch_size=64)
z_test = encoder.predict(x_test, batch_size=64)

print("z_train:", z_train.shape)
print("z_valid:", z_valid.shape)
print("z_test:", z_test.shape)

# save latent splits
def save_latent(filename, z, y, subjects):
    f = h5py.File(os.path.join(loadPath, filename), "w")
    f.create_dataset("latent", data=z.astype(np.float32))
    f.create_dataset("tasks", data=y)
    f.create_dataset("subjects", data=subjects)
    f.close()

save_latent("train1800_latent.h5", z_train, ytr, tr_subjects)
save_latent("valid360_latent.h5", z_valid, yval, val_subjects)
save_latent("test360_latent.h5", z_test, yts, ts_subjects)

print("saved latent splits")

I0000 00:00:1774223241.979254  122981 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774223244.019883  122981 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 898 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1774223244.020444  122981 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21865 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
I0000 00:00:1774223245.504767  123128 service.cc:153] XLA service 0x7a32d40b16e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774223245.504789  123128 service.cc:161]   StreamExec

19/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

I0000 00:00:1774223246.489643  123128 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
z_train: (1800, 128)
z_valid: (360, 128)
z_test: (360, 128)
saved latent splits
